# 02. Phân cụm Phim bằng K-Means trên ALS Latent Item Factors (Bonus)
**Phụ trách**: Nguyễn Văn Tâm (2570358)
**Mục tiêu**: Trích xuất vector đặc trưng ẩn `itemFactors` từ mô hình Spark ALS và áp dụng thuật toán phân cụm K-Means để gom nhóm các bộ phim có phong cách tương đồng.

**Ý tưởng**: ALS biểu diễn mỗi phim bằng một vector `rank` chiều học từ hành vi chấm điểm. Hai phim gần nhau trong không gian này là hai phim được *cùng nhóm người dùng* yêu thích. K-Means trên các vector đó gom được các nhóm phim tương đồng **mà không cần dùng thể loại** — thể loại (`genres`) chỉ được dùng *sau* để diễn giải từng cụm.

**Các bước**:
1. Nạp mô hình ALS đã lưu (hoặc huấn luyện nhanh nếu chưa có).
2. Lấy `itemFactors`, chỉ giữ phim có đủ lượt đánh giá (vector ít nhiễu).
3. Chọn số cụm `k` bằng Silhouette và Elbow (WSSSE).
4. Phân cụm, diễn giải từng cụm qua thể loại nổi bật và phim tiêu biểu.
5. Trực quan hóa 2D bằng PCA.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.common.config import load_config
from src.common.spark_session import get_spark_session

cfg = load_config('dev')
spark = get_spark_session(cfg)

## 1. Nạp dữ liệu & mô hình ALS

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

from src.common.config import PROJECT_ROOT
from src.ml.clustering import extract_item_factors, fit_kmeans, profile_clusters, scan_k
from src.ml.train_als import load_als_model, load_and_split_ratings, save_als_model, train_als_model

ENV = 'dev'
ratings_df = spark.read.parquet(str(PROJECT_ROOT / cfg['paths']['ratings_parquet']))
movies_df = spark.read.parquet(str(PROJECT_ROOT / cfg['paths']['movies_parquet']))

model_path = PROJECT_ROOT / cfg['paths']['als_model']
if model_path.exists():
    als_model = load_als_model(ENV)
    print(f'Đã nạp mô hình ALS từ {model_path}')
else:
    print('Chưa có mô hình đã lưu -> huấn luyện nhanh trên tập train ...')
    _, train_df, _ = load_and_split_ratings(ENV)
    als_model, _ = train_als_model(train_df, ENV)
    save_als_model(als_model, ENV)

print('Số chiều latent (rank):', als_model.rank)

## 2. Trích xuất `itemFactors`

In [ ]:
cl_cfg = cfg.get('clustering', {})
SEED = int(cfg['project']['seed'])
MIN_RATINGS = int(cl_cfg.get('min_item_ratings', 20))

factors_df = extract_item_factors(als_model, ratings_df, MIN_RATINGS).cache()
print(f'Số phim có >= {MIN_RATINGS} lượt đánh giá dùng để phân cụm: {factors_df.count():,}')
factors_df.show(3, truncate=80)

## 3. Chọn số cụm `k`
- **Silhouette** (càng gần 1 càng tốt): độ tách biệt giữa các cụm.
- **WSSSE** (Elbow method): tổng bình phương khoảng cách tới tâm cụm, chọn điểm "khuỷu tay" nơi đường cong bắt đầu thoải.

In [ ]:
K_VALUES = cl_cfg.get('k_values', [5, 8, 10, 12, 15])
scan_df = pd.DataFrame(scan_k(factors_df, K_VALUES, SEED))
display(scan_df)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(scan_df['k'], scan_df['wssse'], 'o-', color='tab:blue', label='WSSSE')
ax1.set_xlabel('Số cụm k')
ax1.set_ylabel('WSSSE', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(scan_df['k'], scan_df['silhouette'], 's--', color='tab:orange', label='Silhouette')
ax2.set_ylabel('Silhouette', color='tab:orange')
plt.title('Chọn k cho K-Means trên ALS itemFactors')
fig.tight_layout()

figures_dir = PROJECT_ROOT / cfg['paths']['figures_dir']
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / 'kmeans_choose_k.png', dpi=150)
plt.show()

## 4. Phân cụm & diễn giải từng cụm

In [ ]:
K = int(cl_cfg.get('k', 10))   # có thể đổi theo kết quả Silhouette / Elbow ở trên
kmeans_model, clustered_df = fit_kmeans(factors_df, K, SEED)

profile_pdf = profile_clusters(clustered_df, movies_df, ratings_df).toPandas()
pd.set_option('display.max_colwidth', None)
display(profile_pdf)

In [ ]:
# Lưu nhãn cụm phim để dùng lại (báo cáo / demo)
out_path = str(PROJECT_ROOT / cfg['paths']['clusters_parquet'])
clustered_df.select('movie_id', 'cluster').write.mode('overwrite').parquet(out_path)
print('Đã lưu nhãn cụm tại', out_path)

## 5. Trực quan hóa 2D bằng PCA

In [ ]:
from pyspark.ml.feature import PCA
from pyspark.ml.functions import vector_to_array

pca_model = PCA(k=2, inputCol='features', outputCol='pca').fit(clustered_df)
print('Tỷ lệ phương sai giải thích bởi 2 thành phần:', [round(v, 3) for v in pca_model.explainedVariance])

plot_pdf = (
    pca_model.transform(clustered_df)
    .select('cluster', vector_to_array('pca').alias('xy'))
    .sample(fraction=min(1.0, 5000 / max(clustered_df.count(), 1)), seed=SEED)
    .toPandas()
)
plot_pdf['x'] = plot_pdf['xy'].str[0]
plot_pdf['y'] = plot_pdf['xy'].str[1]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(plot_pdf['x'], plot_pdf['y'], c=plot_pdf['cluster'], cmap='tab20', s=6, alpha=0.7)
plt.legend(*scatter.legend_elements(), title='Cụm', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'Phim trong không gian ALS itemFactors (PCA 2D, k={K})')
plt.tight_layout()
plt.savefig(figures_dir / 'kmeans_pca_clusters.png', dpi=150)
plt.show()

## 6. Nhận xét
- Mỗi cụm gom các phim được cùng nhóm khán giả ưa thích; cột `top_genres` cho thấy nhiều cụm trùng khớp tự nhiên với thể loại (ví dụ Animation/Children, Horror/Thriller) dù K-Means **không hề dùng thông tin thể loại**.
- Điều đó chứng tỏ vector tiềm ẩn của ALS học được ngữ nghĩa nội dung phim chỉ từ hành vi chấm điểm.
- Ứng dụng: gợi ý "phim tương tự" trong cùng cụm, đa dạng hóa danh sách Top-N (lấy phim từ nhiều cụm), hoặc gợi ý khởi tạo cho người dùng mới theo cụm sở thích họ chọn.
- Silhouette trên không gian latent thường không cao vì các phim nằm liên tục chứ không tách thành khối rõ rệt — K-Means ở đây dùng để *khám phá và diễn giải*, không phải phân loại tuyệt đối.

In [ ]:
spark.stop()